In [ ]:
import os

os.environ["TORCHINDUCTOR_CACHE_DIR"] = "torch_cache"

#os.environ["TORCH_LOGS"]="recompiles"

os.environ["TORCHINDUCTOR_FX_GRAPH_CACHE"] = "1"
os.environ["TORCHINDUCTOR_AUTOGRAD_CACHE"] = "1"

import cv2
import numpy as np
import torch
import gc

import torch._dynamo.config as cfg

cfg.recompile_limit = float('inf')  # always allow recompile to support any image size
cfg.accumulated_recompile_limit = float('inf') 


from realesrgan.archs.rrdb_pixelshuffle_v2_attn_arch import RRDBNet_pxlshuffle_v2_attn as RRDBNet

scale = 4



def process(img, f):

    img = img.astype(np.float32)
    if np.max(img) > 256:  # 16-bit image
        max_range = 65535
        print('\tInput is a 16-bit image')
    else:
        max_range = 255
    img = img / max_range
    img = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()
    img = img.unsqueeze(0)
    
    
    # pre_pad
    img = torch.nn.functional.pad(img, (5, 5, 5, 5), 'reflect').float()
        
    # mod scale
    mod_scale = 1
    if scale == 2:
        mod_scale = 2
    if scale == 1:
        mod_scale = 4
    
    mod_pad_h, mod_pad_w = 0, 0
    b, c, h, w = img.size()
    if (h % mod_scale != 0):
      mod_pad_h = (mod_scale - h % mod_scale)
    if (w % mod_scale != 0):
        mod_pad_w = (mod_scale - w % mod_scale)
    img = torch.nn.functional.pad(img, (0, mod_pad_w, 0, mod_pad_h), 'reflect')

    img = img.to("cuda")
    #img = img.half()

    import time
    t0 = time.time()
    print("testing...")
    with torch.no_grad(): 
        with torch.amp.autocast('cuda'):
            out = f(img)
    print(time.time() - t0)
    print("Output dtype:", out.dtype)
    
    # remove extra pad
    if mod_scale is not None:
        _, _, h, w = out.size()
        out = out[:, :, 0:h - mod_pad_h * scale, 0:w - mod_pad_w * scale]
    
    # remove prepad
    _, _, h, w = out.size()
    out = out[:, :, 5*scale:h - 5*scale, 5*scale:w - 5*scale]
    
    
    output_img = out.data.squeeze().float().cpu().clamp_(0, 1).numpy()
    output_img = np.transpose(output_img[:, :, :], (1, 2, 0))
    
    if max_range == 65535:  # 16-bit image
        output_img = (output_img * 65535.0).round().astype(np.uint16)
    else:
        output_img = (output_img * 255.0).round().astype(np.uint8)

  

    return output_img
    


In [ ]:
loadnet = torch.load("../Real-ESRGAN/experiments/train_v2_attn_x4_m/models/net_g_440000.pth")
# prefer to use params_ema
if 'params_ema' in loadnet:
    keyname = 'params_ema'
else:
    keyname = 'params'
print(keyname)

In [ ]:

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=32, num_grow_ch=32, num_pre_upscale_ch = 256, scale=scale)
_ = model.eval()
_ = model.to("cuda")
model.load_state_dict(loadnet[keyname], strict=True)
model.compile(dynamic=False, fullgraph=True)

In [ ]:
img = cv2.imread("tests/data/lq_4/comic.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (3100,3100))
print(img.shape)
out = process(img, model)

#del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


In [ ]:
import matplotlib.pyplot as plt
plt.imshow(out)

In [ ]:
for x in [200, 1000,2000,3000,4000]:
    for y in [500,600,1200,3000,4000]:
#for x in [3000,4000]:
#    for y in [3000,4000, 3500,3100,3200,3300]:

        img = cv2.imread("test_image.jpg")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (x,y))
        print (img.shape)
        out = process(img, model)
        
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
